In [ ]:
import numpy as np
import pandas as pd
from obspy import read

xlsx = "/Users/ramonmargarit/Phd/Projects/Heterogeneities_Mantle/Heterogeneities-project/data/processed/Shallow_processed_RESULTS.xlsx"
sheet = "best_7_bands_fixed_hold0"
MSEED_PATH = "/Users/ramonmargarit/Phd/Projects/Heterogeneities_Mantle/Heterogeneities-project/notebooks/All_Shallow_Moonquakes.mseed"

FC = 5.0
BANDS = np.array([3., 4., 5., 6., 7., 8., 9.])
STARTTIME_TOL_S = 2.0
SCENARIOS = [dict(LOWER_TOL=0.75, MIN_POST=4, K_NEG=0, K_PRE_POS=0)]


def load_excel_long(xlsx, sheet, *, FC, BANDS):
    d = pd.read_excel(xlsx, sheet_name=sheet)
    need = ["starttime", "station", "fc_hz", "t0_dt_mean"]
    missing = [c for c in need if c not in d.columns]
    if missing:
        raise KeyError(f"Missing columns in sheet '{sheet}': {missing}")

    d["station"] = d["station"].astype(str)
    d["fc_hz"] = pd.to_numeric(d["fc_hz"], errors="coerce").astype(float)
    d["starttime_dt"] = pd.to_datetime(d["starttime"], errors="coerce", utc=True)
    d["t0_dt_mean_dt"] = pd.to_datetime(d["t0_dt_mean"], errors="coerce", utc=True)

    if "distance" in d.columns:
        d["distance_deg"] = pd.to_numeric(d["distance"], errors="coerce")
    elif "epi_deg" in d.columns:
        d["distance_deg"] = pd.to_numeric(d["epi_deg"], errors="coerce")
    else:
        d["distance_deg"] = np.nan

    d = d[d["fc_hz"].isin(BANDS)].copy()
    d["event"] = d["starttime_dt"].astype(str) + "__" + d["station"]
    ref = (
        d[d["fc_hz"].eq(FC)][["event", "t0_dt_mean_dt"]]
        .rename(columns={"t0_dt_mean_dt": "t0_fc_dt"})
        .groupby("event", as_index=False)["t0_fc_dt"]
        .min()
    )
    d = d.merge(ref, on="event", how="left")
    d["dt_rel"] = (d["t0_dt_mean_dt"] - d["t0_fc_dt"]).dt.total_seconds()
    d = d[d["dt_rel"].notna() & d["starttime_dt"].notna() & d["t0_dt_mean_dt"].notna()].copy()
    return d[["event", "station", "starttime_dt", "fc_hz", "dt_rel", "distance_deg", "t0_dt_mean_dt"]].rename(columns={"fc_hz": "band"})


def build_event_band_matrix(df_long, *, BANDS):
    return (
        df_long.pivot_table(index="event", columns="band", values="dt_rel", aggfunc="first")
        .reindex(columns=BANDS)
        .sort_index()
    )


def select_events(*, dt_mat, FC, BANDS, MIN_POST, K_NEG, K_PRE_POS=0, LOWER_TOL=0.0):
    post_bands = [b for b in BANDS if b > FC]
    pre_bands = [b for b in BANDS if b < FC]
    keep = []
    for ev in dt_mat.index:
        dt = dt_mat.loc[ev]
        post_vals = dt[post_bands].dropna()
        if len(post_vals) < MIN_POST:
            keep.append(False)
            continue
        if int((post_vals <= LOWER_TOL).sum()) > K_NEG:
            keep.append(False)
            continue
        pre_vals = dt[pre_bands].dropna()
        if int((pre_vals > LOWER_TOL).sum()) > K_PRE_POS:
            keep.append(False)
            continue
        keep.append(True)
    return pd.Series(keep, index=dt_mat.index, name="keep")


def match_traces_to_excel_events(st, df_long, tol_s):
    by_sta = {sta: g.copy() for sta, g in df_long.groupby("station")}
    event_to_trace = {}
    for tr in st:
        sta = str(getattr(tr.stats, "station", "")).strip()
        if not sta or sta not in by_sta:
            continue
        tr_t0 = pd.Timestamp(tr.stats.starttime.datetime, tz="UTC")
        g = by_sta[sta]
        dt = (g["starttime_dt"] - tr_t0).dt.total_seconds().abs()
        j = dt.idxmin()
        if not np.isfinite(dt.loc[j]):
            continue
        if dt.loc[j] <= tol_s:
            ev = g.loc[j, "event"]
            if ev in event_to_trace:
                prev_tr, prev_diff = event_to_trace[ev]
                if dt.loc[j] < prev_diff:
                    event_to_trace[ev] = (tr, float(dt.loc[j]))
            else:
                event_to_trace[ev] = (tr, float(dt.loc[j]))
    return {ev: tr for ev, (tr, _) in event_to_trace.items()}


df_long = load_excel_long(xlsx, sheet, FC=FC, BANDS=BANDS)
dt_mat = build_event_band_matrix(df_long, BANDS=BANDS)
t0_mat = (
    df_long.pivot_table(index="event", columns="band", values="t0_dt_mean_dt", aggfunc="first")
    .reindex(columns=BANDS)
    .sort_index()
)
st = read(MSEED_PATH)
event_to_trace = match_traces_to_excel_events(st, df_long, tol_s=STARTTIME_TOL_S)

selected_rows = []
for cfg in SCENARIOS:
    keep_mask = select_events(
        dt_mat=dt_mat,
        FC=FC,
        BANDS=BANDS,
        MIN_POST=int(cfg["MIN_POST"]),
        K_NEG=int(cfg["K_NEG"]),
        K_PRE_POS=int(cfg["K_PRE_POS"]),
        LOWER_TOL=float(cfg["LOWER_TOL"]),
    )
    kept_events = [ev for ev in keep_mask.index[keep_mask].tolist() if ev in event_to_trace]
    for ev in kept_events:
        station = ev.split("__", 1)[-1]
        t0_fc = df_long.loc[(df_long["event"] == ev) & (df_long["band"] == FC), "t0_dt_mean_dt"]
        distance_deg = df_long.loc[df_long["event"] == ev, "distance_deg"].dropna()
        selected_rows.append({
            "event": ev,
            "station": station,
            "time_utc": t0_fc.iloc[0] if not t0_fc.empty else pd.NaT,
            "epi_deg": float(distance_deg.iloc[0]) if not distance_deg.empty else np.nan,
        })

selected_events_df = (
    pd.DataFrame(selected_rows)
    .drop_duplicates(subset=["event"])
    .sort_values(["epi_deg", "time_utc"], na_position="last")
    .reset_index(drop=True)
)
print(f"Matched {len(event_to_trace)} Excel events to MiniSEED traces.")
print(selected_events_df.to_string(index=False))

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import butter, sosfiltfilt, hilbert, get_window

RESULTS_DIR = os.path.join(os.getcwd(), "results", "gillet2017_envelopes_rise")
EVENT_FIG_DIR = os.path.join(RESULTS_DIR, "event_figures")
os.makedirs(EVENT_FIG_DIR, exist_ok=True)

HALF_BW = 0.5
BP_ORDER = 4
SMOOTH_WINDOW_S = 5.0
NOISE_START = -150.0
NOISE_END = -30.0
ONSET_SEARCH_START = -20.0
ONSET_SEARCH_END = 120.0
RISE_WINDOW_S = 30.0
ONSET_SIGMA = 3.0
MIN_NOISE_N = 40
MIN_ONSET_SEC = 5.0
MAX_PLOT_TIME = 500.0
RESULTS_CSV = os.path.join(RESULTS_DIR, "gillet2017_rise_measurements.csv")


def compute_bandpass_limits(center_hz, half_width_hz, sampling_hz):
    low_hz = max(center_hz - half_width_hz, 0.001)
    high_hz = min(center_hz + half_width_hz, 0.99 * sampling_hz / 2)
    return low_hz, high_hz


def build_rms_envelope(signal, sampling_hz, low_hz, high_hz, smooth_window_s, filter_order):
    sos = butter(filter_order, [low_hz / (sampling_hz / 2), high_hz / (sampling_hz / 2)], btype="bandpass", output="sos")
    bandpassed = sosfiltfilt(sos, signal)
    analytic_env = np.abs(hilbert(bandpassed))
    nwin = int(round(smooth_window_s * sampling_hz)) | 1
    window = get_window("hann", nwin)
    window = window / window.sum()
    return np.sqrt(np.convolve(analytic_env ** 2, window, mode="same"))


def estimate_noise_stats(time_s, envelope, noise_start, noise_end, min_noise_n):
    mask = np.isfinite(time_s) & np.isfinite(envelope) & (envelope > 0) & (time_s >= noise_start) & (time_s <= noise_end)
    if mask.sum() < min_noise_n:
        raise RuntimeError("Not enough points in pre-event noise window")
    noise_vals = envelope[mask]
    return float(np.median(noise_vals)), float(np.std(noise_vals, ddof=0))


def detect_onset(time_s, envelope, *, threshold, onset_search_start, onset_search_end, min_onset_sec, sampling_hz):
    mask = np.isfinite(time_s) & np.isfinite(envelope) & (time_s >= onset_search_start) & (time_s <= onset_search_end)
    if mask.sum() == 0:
        raise RuntimeError("No samples in onset search window")
    t_sel = time_s[mask]
    e_sel = envelope[mask]
    above = e_sel >= threshold
    n_consecutive = max(1, int(round(min_onset_sec * sampling_hz)))
    run = 0
    for idx, is_above in enumerate(above):
        run = run + 1 if is_above else 0
        if run >= n_consecutive:
            return float(t_sel[idx - n_consecutive + 1])
    raise RuntimeError("No onset crossing found")


def find_tmax_in_window(time_s, envelope, onset_s, rise_window_s):
    rise_end_s = onset_s + rise_window_s
    peak_mask = np.isfinite(time_s) & np.isfinite(envelope) & (time_s >= onset_s) & (time_s <= rise_end_s)
    if peak_mask.sum() == 0:
        raise RuntimeError("No samples in fixed rise window")
    t_peak = time_s[peak_mask]
    e_peak = envelope[peak_mask]
    peak_idx = int(np.argmax(e_peak))
    return float(t_peak[peak_idx]), float(e_peak[peak_idx]), float(rise_end_s)


def fit_rise_line(time_s, envelope, onset_s, rise_end_s):
    mask = np.isfinite(time_s) & np.isfinite(envelope) & (envelope > 0) & (time_s >= onset_s) & (time_s <= rise_end_s)
    if mask.sum() < 3:
        raise RuntimeError("Not enough points in fixed rise window")
    t_fit = time_s[mask]
    y_fit = np.log(envelope[mask])
    slope, intercept = np.polyfit(t_fit, y_fit, 1)
    y_model = intercept + slope * t_fit
    rmse_log = float(np.sqrt(np.mean((y_model - y_fit) ** 2)))
    env_model = np.exp(y_model)
    return float(slope), float(intercept), rmse_log, t_fit, env_model


def plot_event_gillet_rise(event_id, station, epicentral_deg, band_rows, output_dir):
    nrows = len(band_rows)
    fig, axes = plt.subplots(nrows, 2, figsize=(14, max(3.0 * nrows, 4.5)), sharex=True)
    if nrows == 1:
        axes = np.array([axes])
    for ax_row, row in zip(axes, band_rows):
        ax_lin, ax_log = ax_row
        ax_lin.plot(row["time_s"], row["envelope"], color="k", lw=1)
        ax_lin.axhline(row["threshold"], color="tab:red", ls=":")
        ax_lin.axvline(row["onset_s"], color="tab:blue", ls="--")
        ax_lin.axvline(row["tmax_s"], color="tab:green", ls="--")
        ax_lin.axvline(row["rise_end_s"], color="tab:purple", ls="--")
        ax_lin.plot(row["t_rise_model"], row["env_rise_model"], color="crimson", lw=2)
        ax_lin.set_xlim(-100, MAX_PLOT_TIME)
        ax_lin.set_ylabel(f"{row['fc_Hz']:.1f} Hz")
        ax_lin.set_title(f"Linear | onset={row['onset_s']:.1f}s | tmax={row['tmax_s']:.1f}s | win={row['rise_end_s'] - row['onset_s']:.1f}s")
        ax_lin.grid(True, alpha=0.3)

        ax_log.semilogy(row["time_s"], row["envelope"], color="k", lw=1)
        ax_log.semilogy(row["t_rise_model"], row["env_rise_model"], color="crimson", lw=2)
        ax_log.axhline(row["threshold"], color="tab:red", ls=":")
        ax_log.axvline(row["onset_s"], color="tab:blue", ls="--")
        ax_log.axvline(row["tmax_s"], color="tab:green", ls="--")
        ax_log.axvline(row["rise_end_s"], color="tab:purple", ls="--")
        ax_log.set_xlim(-100, MAX_PLOT_TIME)
        ax_log.set_title(f"Semilogy | fc={row['fc_Hz']:.1f} Hz | rmse={row['rmse_log']:.3f}")
        ax_log.grid(True, which="both", alpha=0.3)
    axes[-1, 0].set_xlabel("Time since excel_t0 (s)")
    axes[-1, 1].set_xlabel("Time since excel_t0 (s)")
    fig.suptitle(f"Gillet-style fixed-window rising analysis | {event_id} | station={station} | distance={epicentral_deg:.2f} deg")
    fig.tight_layout()
    safe_event = "".join(ch if ch.isalnum() else "_" for ch in event_id)
    out_path = os.path.join(output_dir, f"{safe_event}_gillet2017_rise.png")
    fig.savefig(out_path, dpi=200)
    plt.show()
    return out_path


measurements = []
failed_rows = []

for event_idx, row in selected_events_df.iterrows():
    ev = row["event"]
    tr = event_to_trace.get(ev)
    if tr is None:
        failed_rows.append({"event": ev, "fc_Hz": np.nan, "reason": "Missing MiniSEED trace"})
        continue

    tr_start_utc = pd.Timestamp(tr.stats.starttime.datetime, tz="UTC")
    fs = float(tr.stats.sampling_rate)
    signal = tr.data.astype(float)
    t_raw = np.arange(signal.size) / fs
    band_rows = []

    print(f"Processing event {event_idx + 1}/{len(selected_events_df)}: {ev}")

    for fc_band in BANDS:
        try:
            t0_utc = t0_mat.loc[ev, fc_band]
        except KeyError:
            t0_utc = pd.NaT
        if pd.isna(t0_utc):
            failed_rows.append({"event": ev, "fc_Hz": float(fc_band), "reason": "Missing t0_dt_mean"})
            continue

        t0_utc = pd.Timestamp(t0_utc)
        t0_utc = t0_utc.tz_localize("UTC") if t0_utc.tzinfo is None else t0_utc.tz_convert("UTC")
        t0_offset_s = (t0_utc - tr_start_utc).total_seconds()
        time_s = t_raw - t0_offset_s

        fl_hz, fu_hz = compute_bandpass_limits(fc_band, HALF_BW, fs)
        envelope = build_rms_envelope(signal, fs, fl_hz, fu_hz, SMOOTH_WINDOW_S, BP_ORDER)
        if envelope.size != time_s.size:
            i0 = max(0, (time_s.size - envelope.size) // 2)
            time_s = time_s[i0:i0 + envelope.size]

        try:
            noise_median, noise_std = estimate_noise_stats(time_s, envelope, NOISE_START, NOISE_END, MIN_NOISE_N)
            threshold = noise_median + ONSET_SIGMA * noise_std
            onset_s = detect_onset(
                time_s,
                envelope,
                threshold=threshold,
                onset_search_start=ONSET_SEARCH_START,
                onset_search_end=ONSET_SEARCH_END,
                min_onset_sec=MIN_ONSET_SEC,
                sampling_hz=fs,
            )
            tmax_s, emax, rise_end_s = find_tmax_in_window(time_s, envelope, onset_s, RISE_WINDOW_S)
            if tmax_s < onset_s:
                raise RuntimeError("Peak occurs before onset")
            slope_log, intercept_log, rmse_log, t_rise_model, env_rise_model = fit_rise_line(time_s, envelope, onset_s, rise_end_s)
        except Exception as exc:
            failed_rows.append({"event": ev, "fc_Hz": float(fc_band), "reason": str(exc)})
            continue

        trise_s = float(tmax_s - onset_s)
        band_rows.append({
            "fc_Hz": float(fc_band),
            "time_s": time_s.copy(),
            "envelope": envelope.copy(),
            "threshold": threshold,
            "onset_s": onset_s,
            "tmax_s": tmax_s,
            "rise_end_s": rise_end_s,
            "trise_s": trise_s,
            "rmse_log": rmse_log,
            "t_rise_model": t_rise_model,
            "env_rise_model": env_rise_model,
        })
        measurements.append({
            "event": ev,
            "station": row["station"],
            "epi_deg": float(row["epi_deg"]),
            "event_time_utc": row["time_utc"],
            "fc_Hz": float(fc_band),
            "noise_median": noise_median,
            "noise_std": noise_std,
            "threshold": threshold,
            "onset_s": onset_s,
            "tmax_s": tmax_s,
            "rise_end_s": rise_end_s,
            "trise_s": trise_s,
            "emax": emax,
            "slope_log": slope_log,
            "intercept_log": intercept_log,
            "rmse_log": rmse_log,
        })

    if band_rows:
        plot_event_gillet_rise(ev, row["station"], row["epi_deg"], band_rows, EVENT_FIG_DIR)

measurements_df = pd.DataFrame(measurements)
failed_fits_df = pd.DataFrame(failed_rows)
if measurements_df.empty:
    print("No successful Gillet-style rise measurements were produced.")
else:
    measurements_df = measurements_df.sort_values(["event_time_utc", "fc_Hz"]).reset_index(drop=True)
    measurements_df.to_csv(RESULTS_CSV, index=False)
    summary_cols = ["event", "station", "epi_deg", "fc_Hz", "onset_s", "tmax_s", "rise_end_s", "trise_s", "rmse_log"]
    print(measurements_df[summary_cols].to_string(index=False))
    print()
    print(f"Saved measurements to: {RESULTS_CSV}")
if not failed_fits_df.empty:
    print()
    print("Failed measurements:")
    print(failed_fits_df.to_string(index=False))

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RESULTS_DIR = os.path.join(os.getcwd(), "results", "gillet2017_envelopes_rise")
SUMMARY_CSV = os.path.join(RESULTS_DIR, "gillet2017_rise_summary_by_band.csv")

if measurements_df.empty:
    raise RuntimeError("Run the Gillet-style rise processing cell successfully before the summary analysis.")

print("Summary by event and band:")
print(measurements_df[["event", "station", "epi_deg", "fc_Hz", "onset_s", "tmax_s", "rise_end_s", "trise_s", "rmse_log"]].to_string(index=False))

print()
print("Summary by frequency band:")
summary_by_band = (
    measurements_df.groupby("fc_Hz")[["onset_s", "tmax_s", "trise_s"]]
    .agg(["count", "mean", "std", "median", "min", "max"])
)
summary_by_band.to_csv(SUMMARY_CSV)
print(summary_by_band.to_string())

print()
print("Linear fits versus epicentral distance by band:")
for fc_band in sorted(measurements_df["fc_Hz"].unique()):
    band_df = measurements_df.loc[measurements_df["fc_Hz"] == fc_band].sort_values("epi_deg")
    if len(band_df) < 2:
        print(f"Band {fc_band:.1f} Hz: not enough events")
        continue
    onset_coef = np.polyfit(band_df["epi_deg"], band_df["onset_s"], 1)
    tmax_coef = np.polyfit(band_df["epi_deg"], band_df["tmax_s"], 1)
    trise_coef = np.polyfit(band_df["epi_deg"], band_df["trise_s"], 1)
    print(f"Band {fc_band:.1f} Hz: onset = {onset_coef[0]:.3f}*D + {onset_coef[1]:.3f}, tmax = {tmax_coef[0]:.3f}*D + {tmax_coef[1]:.3f}, trise = {trise_coef[0]:.3f}*D + {trise_coef[1]:.3f}")

    d_plot = np.linspace(band_df["epi_deg"].min(), band_df["epi_deg"].max(), 100)
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    axes[0].scatter(band_df["epi_deg"], band_df["onset_s"], color="k")
    axes[0].plot(d_plot, np.polyval(onset_coef, d_plot), color="crimson")
    axes[0].set_title(f"onset | {fc_band:.1f} Hz")
    axes[0].set_xlabel("Epicentral distance (deg)")
    axes[0].set_ylabel("Onset time (s)")
    axes[0].grid(True, alpha=0.3)

    axes[1].scatter(band_df["epi_deg"], band_df["tmax_s"], color="k")
    axes[1].plot(d_plot, np.polyval(tmax_coef, d_plot), color="crimson")
    axes[1].set_title(f"tmax in fixed window | {fc_band:.1f} Hz")
    axes[1].set_xlabel("Epicentral distance (deg)")
    axes[1].set_ylabel("tmax (s)")
    axes[1].grid(True, alpha=0.3)

    axes[2].scatter(band_df["epi_deg"], band_df["trise_s"], color="k")
    axes[2].plot(d_plot, np.polyval(trise_coef, d_plot), color="crimson")
    axes[2].set_title(f"trise | {fc_band:.1f} Hz")
    axes[2].set_xlabel("Epicentral distance (deg)")
    axes[2].set_ylabel("Rise time (s)")
    axes[2].grid(True, alpha=0.3)

    fig.tight_layout()
    plt.show()

print()
print(f"Saved band summary to: {SUMMARY_CSV}")